# Aula 13B · APIs REST

**Noite B — o problema.** Nenhum conceito novo: esta noite é laboratório sobre o
[capítulo 13 do
site](https://lacouth.github.io/python_telecom-site/unidade7-redes/13-apis-rest/),
que a [noite
A](https://colab.research.google.com/github/lacouth/python_telecom-site/blob/main/notebooks/aula13a-apis-rest.ipynb)
apresentou. Faltou à noite A? Cada exercício traz o link 📖 para a seção que ele usa
— e o caderno da noite A tem os passos.

**Roteiro** (1h40): 🔥 aquecimento · ⚠️ a regra da noite · 🎯 prática · 📟 resolvendo o
chamado · 📋 a Lista 13, começada aqui · 🚪 antes de sair

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🎯 Sua vez:** escreva a solução e rode a célula `confere` logo abaixo dela —
  ✅ quer dizer que acertou, ❌ mostra o que ainda falta. A dica está recolhida:
  tente antes de abrir.
- Travou? Antes da dica, abra o link 📖 do exercício: a seção do capítulo que
  ele usa tem o exemplo completo.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler agora (usa coisas que só veremos mais tarde).
import math


def _mostra(argumentos):
    return ", ".join(repr(a) for a in argumentos)


def _igual(veio, esperado):
    if isinstance(esperado, float) and isinstance(veio, (int, float)):
        return math.isclose(veio, esperado, abs_tol=1e-9)
    return veio == esperado


def confere(funcao, casos):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if _igual(veio, esperado):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if _igual(valor, esperado):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")


# --- dados desta aula ---
# simulador: API REST da Maré Net — não precisa ler (é o "servidor" das aulas).
# Sobe um servidor HTTP nesta própria sessão, em http://127.0.0.1:8765, que
# responde em JSON como a API de um sistema de inventário de rede.
import json
import threading
import time
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

PORTA_API = 8765

_EQUIPAMENTOS = [
    {"nome": "OLT-CENTRO-01", "tipo": "OLT", "ip": "10.0.1.10", "localidade": "Centro", "em_servico": True},
    {"nome": "OLT-NORTE-02", "tipo": "OLT", "ip": "10.0.2.10", "localidade": "Zona Norte", "em_servico": True},
    {"nome": "OLT-SUL-03", "tipo": "OLT", "ip": "10.0.3.10", "localidade": "Zona Sul", "em_servico": False},
    {"nome": "SWITCH-CENTRO-01", "tipo": "SWITCH", "ip": "10.0.1.20", "localidade": "Centro", "em_servico": True},
    {"nome": "SWITCH-NORTE-02", "tipo": "SWITCH", "ip": "10.0.2.20", "localidade": "Zona Norte", "em_servico": True},
    {"nome": "RADIO-OESTE-01", "tipo": "RADIO", "ip": "10.0.5.10", "localidade": "Zona Oeste", "em_servico": True},
]
_ALARMES = [
    {"id": 101, "equipamento": "OLT-CENTRO-01", "severidade": "CRITICAL", "mensagem": "perda de sinal na porta GPON0/1/3"},
    {"id": 102, "equipamento": "SWITCH-NORTE-02", "severidade": "ERROR", "mensagem": "Interface Gi0/12, changed state to down"},
    {"id": 103, "equipamento": "RADIO-OESTE-01", "severidade": "WARNING", "mensagem": "enlace degradado"},
    {"id": 104, "equipamento": "OLT-CENTRO-01", "severidade": "WARNING", "mensagem": "temperatura acima do limite"},
    {"id": 105, "equipamento": "RADIO-OESTE-01", "severidade": "CRITICAL", "mensagem": "enlace fora do ar"},
]
_chamadas_instavel = [0]


class _Tratador(BaseHTTPRequestHandler):
    def log_message(self, *args):          # sem log na tela
        pass

    def _responde(self, status, corpo):
        dados = json.dumps(corpo, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(dados)))
        self.end_headers()
        self.wfile.write(dados)

    def do_GET(self):
        url = urlparse(self.path)
        partes = [p for p in url.path.split("/") if p]
        filtros = {chave: valores[0] for chave, valores in parse_qs(url.query).items()}
        if partes == ["api", "saude"]:
            return self._responde(200, {"status": "ok"})
        if partes == ["api", "equipamentos"]:
            itens = [e for e in _EQUIPAMENTOS
                     if all(str(e.get(k)) == v for k, v in filtros.items())]
            return self._responde(200, {"count": len(itens), "results": itens})
        if len(partes) == 3 and partes[:2] == ["api", "equipamentos"]:
            for e in _EQUIPAMENTOS:
                if e["nome"] == partes[2]:
                    return self._responde(200, e)
            return self._responde(404, {"erro": f"equipamento {partes[2]} não encontrado"})
        if partes == ["api", "alarmes"]:
            itens = [a for a in _ALARMES
                     if all(str(a.get(k)) == v for k, v in filtros.items())]
            return self._responde(200, {"count": len(itens), "results": itens})
        if partes == ["api", "lento"]:
            time.sleep(3)
            return self._responde(200, {"status": "finalmente"})
        if partes == ["api", "instavel"]:
            _chamadas_instavel[0] += 1
            if _chamadas_instavel[0] % 2 == 1:
                return self._responde(503, {"erro": "serviço temporariamente indisponível"})
            return self._responde(200, {"status": "ok"})
        return self._responde(404, {"erro": f"rota {url.path} não existe"})


def inicia_api(porta=PORTA_API):
    """Sobe a API simulada (se ainda não estiver no ar) e devolve o endereço."""
    endereco = f"http://127.0.0.1:{porta}"
    try:
        urllib.request.urlopen(endereco + "/api/saude", timeout=1)
        return endereco                     # já estava no ar
    except OSError:
        pass
    servidor = ThreadingHTTPServer(("127.0.0.1", porta), _Tratador)
    threading.Thread(target=servidor.serve_forever, daemon=True).start()
    return endereco


API = inicia_api()
print("API simulada no ar em", API)

## 🔥 Aquecimento — a pergunta que ficou

No fim da noite A ficou uma pergunta sem resposta. Responda de cabeça, sem rodar.

`requests.get(url)` para um recurso que não existe:
a) levanta erro na hora  b) devolve uma resposta com código 404  c) devolve `None`
d) trava

<details>
<summary><b>Resposta da 1</b></summary>

**b**. A resposta chega; conferir o código é tarefa do script (ou do
`raise_for_status()`).

</details>

## ⚠️ A regra da noite

**Resposta chegou não quer dizer que deu certo.** O `requests.get` devolve uma resposta também para o `404` e o `503`: conferir o
`status_code` é tarefa do script. E as outras duas regras de toda chamada:

- sempre com `timeout=`, para a API que não responde não travar o script;
- numa consulta em lote, um `try` **por item**: a falha em um equipamento não pode
  apagar o painel dos outros.

## 🎯 Prática

Nenhum passo novo: cada exercício usa uma seção da noite A, indicada no link 📖
logo acima dele. Do mais simples para o mais completo.

Vem do bloco *3. Filtrar no servidor: parâmetros* da noite A, e os passos citados na dica são de lá: 📖 [capítulo 13 · Filtrar no servidor: parâmetros](https://lacouth.github.io/python_telecom-site/unidade7-redes/13-apis-rest/#filtrar-no-servidor-parametros)

### 🎯 Sua vez — Quantos de um tipo

Escreva `quantos_do_tipo(api, tipo)`, que pede à API só os equipamentos daquele tipo e
devolve o `count` da resposta.

In [ ]:
def quantos_do_tipo(api, tipo):
    # sua solução aqui
    pass

In [ ]:
confere(quantos_do_tipo, [
    ((API, "OLT"), 3),
    ((API, "SWITCH"), 2),
    ((API, "ROTEADOR"), 0),
])

<details>
<summary><b>💡 Dica</b></summary>

É o passo acima, com `tipo` no lugar de `"OLT"`.

</details>

Vem do bloco *4. O código de status* da noite A, e os passos citados na dica são de lá: 📖 [capítulo 13 · O código de status](https://lacouth.github.io/python_telecom-site/unidade7-redes/13-apis-rest/#o-codigo-de-status)

### 🎯 Sua vez — Onde está o problema?

Escreva `classifica_status(codigo)`, que devolve `"sucesso"` para códigos de 200 a 299,
`"erro no pedido"` de 400 a 499 e `"erro no servidor"` de 500 a 599 (e `"outro"` para o
resto).

In [ ]:
def classifica_status(codigo):
    # sua solução aqui
    pass

In [ ]:
confere(classifica_status, [
    ((200,), "sucesso"),
    ((404,), "erro no pedido"),
    ((401,), "erro no pedido"),
    ((503,), "erro no servidor"),
    ((301,), "outro"),
])

<details>
<summary><b>💡 Dica</b></summary>

Faixas com `if`/`elif`: `200 <= codigo <= 299`, e assim por diante. Ou, se preferir,
`codigo // 100` dá o primeiro algarismo (Aula 01).

</details>

## 📟 Resolvendo o chamado

O chamado que a noite A abriu:

> **Chamado #1502 — NOC Maré Net**
>
> *"Estagiário, o sistema de inventário agora tem uma API. Quero um **painel
> simples**: para cada equipamento **em serviço**, quantos alarmes **críticos** ele
> tem abertos. Hoje alguém entra na tela do sistema e conta um por um."*

### 🎯 Sua vez — O painel de críticos

Escreva `painel(api)`, que devolve um dicionário **nome → quantidade de alarmes
`CRITICAL`**, para cada equipamento **em serviço** (inclusive os que têm zero).

In [ ]:
def painel(api):
    # sua solução aqui
    pass

In [ ]:
confere(painel, [
    ((API,), {"OLT-CENTRO-01": 1, "OLT-NORTE-02": 0, "SWITCH-CENTRO-01": 0,
              "SWITCH-NORTE-02": 0, "RADIO-OESTE-01": 1}),
])

<details>
<summary><b>💡 Dica</b></summary>

Duas consultas: `/api/equipamentos` (para saber quem está em serviço) e, para cada um,
`/api/alarmes` com `params={"equipamento": nome, "severidade": "CRITICAL"}` — o
`count` da resposta é a quantidade.

</details>

**Resposta ao chamado:** a OLT-CENTRO-01 e o RADIO-OESTE-01 têm um alarme crítico cada;
os outros equipamentos em serviço estão limpos. A OLT-SUL-03, fora de serviço, nem
entra no painel. E o script responde em um segundo, sempre do mesmo jeito.

## 📋 A lista, começada aqui

Abra a [Lista 13](https://lacouth.github.io/python_telecom-site/listas/lista13/) —
ou direto o [caderno dela no
Colab](https://colab.research.google.com/github/lacouth/python_telecom-site/blob/main/notebooks/lista13.ipynb)
— e rode a célula de preparo. O **exercício 1** a turma faz junto, respondendo às
perguntas abaixo **antes** de escrever; do 2 em diante, cada um no seu ritmo, com o
professor circulando.

**Exercício 1 — `status_de`**

**1.** Qual é a URL?

<details>
<summary><b>Resposta da 1</b></summary>

`api + caminho`: o endereço da API simulada, que a célula de preparo põe no ar, mais o caminho do recurso.

</details>

**2.** Como fazer a chamada?

<details>
<summary><b>Resposta da 2</b></summary>

`requests.get(url, timeout=5)`, guardando a resposta numa variável.

</details>

**3.** O que devolver?

<details>
<summary><b>Resposta da 3</b></summary>

O `status_code` da resposta — não o `.json()`. O `404` também tem corpo, e é o código que diz o que aconteceu.

</details>

## 🚪 Antes de sair

Responda de cabeça, sem rodar.

**1.** Um código `404` começa com 4. Isso diz que o problema está:
a) no servidor  b) no pedido  c) na rede  d) no JSON

<details>
<summary><b>Resposta da 1</b></summary>

**b**. Começou com 4, o pedido está errado (aqui: o recurso não existe); com 5, o problema é do servidor.

</details>

**2.** `requests.get(url, params={"tipo": "OLT"}, timeout=5)`:
a) filtra no servidor  b) baixa tudo e filtra no script  c) muda o cabeçalho  d) nada

<details>
<summary><b>Resposta da 2</b></summary>

**a**. O filtro vai na URL, e a resposta já chega só com as OLTs.

</details>

## 🏠 Para casa

- [Lista 13](https://lacouth.github.io/python_telecom-site/listas/lista13/) — APIs REST,
  com testes automáticos no Colab.
- Releia o [capítulo 13 do site](https://lacouth.github.io/python_telecom-site/unidade7-redes/13-apis-rest/).
- **Na próxima noite A:** mini-teste sobre esta aula (código de status, `timeout` e
  falha por item).